# 07b — Wiki Training Pairs

Reads every page from the ARO wiki and uses the local model to generate
instruction/answer training pairs from each page.

**Strategy:**
- Split each page into sections by heading (##, ###)
- For each section, ask the model to generate 5 targeted Q&A pairs
- Pairs cover: concept questions, syntax questions, "how do I…" questions,
  "what is the difference between…" questions, and code-completion questions
- Output is validated (non-empty instruction + answer, answer must contain ARO)
- Resume-safe: already-processed sections are skipped

**Wiki source:** cloned from `git@git.ausdertechnik.de:arolang/aro.wiki.git`
(re-clone with the cell below if the local copy is stale)

**Input:**  `../data/wiki/` (markdown files)
**Output:** `../data/07b_wiki/wiki_pairs.jsonl`
            merged into `../data/02_knowledge/knowledge_pairs.jsonl`

## Setup & clone wiki

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, subprocess, hashlib
from pathlib import Path
from collections import defaultdict

DATA_OUT  = GLOBAL_OUT_DIR / '../data/07b_wiki'
OWN_FILE  = DATA_OUT / 'wiki_pairs.jsonl'

DATA_OUT.mkdir(parents=True, exist_ok=True)
WIKI_DIR.mkdir(parents=True, exist_ok=True)

def clone_or_update_wiki():
    git_dir = WIKI_DIR / '.git'
    if git_dir.exists():
        result = subprocess.run(
            ['git', '-C', str(WIKI_DIR), 'pull', '--ff-only'],
            capture_output=True, text=True
        )
        print(f'Wiki updated: {result.stdout.strip() or result.stderr.strip()}')
    else:
        subprocess.run(
            ['git', 'clone', GITLAB_WIKI, str(WIKI_DIR)],
            check=True
        )
        # GitLab wiki uses 'master' branch
        subprocess.run(['git', '-C', str(WIKI_DIR), 'checkout', 'master'],
                       capture_output=True)
        print(f'Wiki cloned to {WIKI_DIR}')

clone_or_update_wiki()

pages = sorted(WIKI_DIR.glob('*.md'))
# Skip sidebar — it's navigation only
pages = [p for p in pages if p.name != '_Sidebar.md']
print(f'Wiki pages: {len(pages)}')
for p in pages:
    print(f'  {p.name}')

## Section splitter

Split each page into sections so each chunk fits in the model's context window.

In [ ]:
MAX_SECTION_CHARS = 3000   # ~750 tokens — leaves room for prompt + response

def split_into_sections(path):
    """
    Returns list of dicts: { page, heading, content, section_id }
    Merges short consecutive sections so we don't generate pairs for 3-line stubs.
    """
    text     = Path(path).read_text()
    page     = Path(path).stem
    heading_re = re.compile(r'^(#{1,3} .+)', re.MULTILINE)

    parts = heading_re.split(text)
    # parts[0] = content before first heading (page intro)
    # parts[1,2] = heading, body; parts[3,4] = heading, body; ...

    sections = []

    # Include the page intro if substantial
    intro = parts[0].strip()
    if len(intro) > 100:
        sections.append({
            'page':    page,
            'heading': page.replace('-', ' '),
            'content': intro[:MAX_SECTION_CHARS],
        })

    i = 1
    pending_heading = ''
    pending_content = ''

    while i < len(parts):
        heading = re.sub(r'^#+\s*', '', parts[i]).strip()
        body    = parts[i + 1].strip() if i + 1 < len(parts) else ''
        i += 2

        # Accumulate short sections together
        combined = (pending_content + '\n\n' + body).strip()
        if len(combined) < 200 and i < len(parts):
            pending_heading = pending_heading or heading
            pending_content = combined
            continue

        # Flush
        if pending_content:
            sections.append({
                'page':    page,
                'heading': pending_heading,
                'content': (pending_content + '\n\n' + body).strip()[:MAX_SECTION_CHARS],
            })
            pending_heading = ''
            pending_content = ''
        elif len(body) >= 100:
            sections.append({
                'page':    page,
                'heading': heading,
                'content': body[:MAX_SECTION_CHARS],
            })

    if pending_content:
        sections.append({
            'page':    page,
            'heading': pending_heading,
            'content': pending_content[:MAX_SECTION_CHARS],
        })

    # Add a stable section_id for resume tracking
    for s in sections:
        raw = f"{s['page']}::{s['heading']}"
        s['section_id'] = hashlib.md5(raw.encode()).hexdigest()[:12]

    return sections


# Preview
all_sections = []
for page in pages:
    secs = split_into_sections(page)
    all_sections.extend(secs)

print(f'Total sections across {len(pages)} pages: {len(all_sections)}')
print(f'  Avg section length: {sum(len(s["content"]) for s in all_sections) // len(all_sections)} chars')
print(f'  Estimated pairs:    {len(all_sections) * 5}')

## Load model

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

# Use load_model() which auto-loads the warm-start adapter from knowledge.json
# and returns (model, tokenizer, load_fn, generate_fn, make_sampler_fn)
model, tokenizer, _, mlx_generate, make_sampler = load_model(kb=kb)
print('Model ready.')

## Pair generation prompt

The model receives a wiki section and returns a JSON array of 5 Q&A pairs.

In [ ]:
SYSTEM_PROMPT = """You are generating training data for a language model that learns the ARO programming language.

You will receive a section from the ARO documentation wiki.
Your task: generate exactly 5 instruction/answer pairs that a developer learning ARO would find useful.

Guidelines:
- Mix question types: concept questions, syntax questions, "how do I..." questions, code completion, comparisons
- Answers must be accurate and directly grounded in the provided documentation
- Include ARO code examples in answers wherever the documentation shows them
- Instructions should be natural questions a developer would ask
- Do NOT invent features not present in the documentation
- Keep answers concise but complete (2-10 sentences + code if applicable)

Output ONLY a valid JSON array — no markdown, no explanation, no preamble:
[
  {"instruction": "...", "answer": "..."},
  {"instruction": "...", "answer": "..."},
  {"instruction": "...", "answer": "..."},
  {"instruction": "...", "answer": "..."},
  {"instruction": "...", "answer": "..."}
]"""

QUESTION_TYPES = [
    'concept',      # What is X? What does X mean?
    'syntax',       # How do you write X in ARO?
    'how_to',       # How do I accomplish Y using ARO?
    'comparison',   # What is the difference between X and Y?
    'code',         # Write an ARO feature set that does Z.
]

def make_generation_prompt(section):
    page_name = section['page'].replace('-', ' ')
    heading   = section['heading']
    content   = section['content']

    question_type_hint = ', '.join(QUESTION_TYPES)

    return (
        f"Wiki page: **{page_name}**\n"
        f"Section: **{heading}**\n\n"
        f"{content}\n\n"
        f"---\n"
        f"Generate 5 instruction/answer training pairs covering these question types: "
        f"{question_type_hint}.\n"
        f"Output only the JSON array."
    )

# Preview prompt for one section
sample = all_sections[3]
print(make_generation_prompt(sample)[:600] + '...')

## Generation + JSON parsing

In [ ]:
MAX_TOKENS  = 1400
MAX_RETRIES = 2

def generate_raw(prompt, temp=0.5):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    return mlx_generate(
        model, tokenizer,
        prompt=tokens,
        max_tokens=MAX_TOKENS,
        sampler=make_sampler(temp=temp),
        verbose=False,
    )


def extract_json_array(text):
    """Extract first JSON array from model output, handling markdown fences."""
    # Strip markdown fences
    text = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.MULTILINE)
    text = re.sub(r'```\s*$', '', text.strip(), flags=re.MULTILINE)
    text = text.strip()

    # Find the outermost [...]
    start = text.find('[')
    if start == -1:
        return None
    depth  = 0
    end    = start
    for i, ch in enumerate(text[start:], start):
        if ch == '[':
            depth += 1
        elif ch == ']':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    try:
        return json.loads(text[start:end])
    except json.JSONDecodeError:
        return None


def validate_pair(pair, page):
    """Returns cleaned pair or None."""
    if not isinstance(pair, dict):
        return None
    instr  = str(pair.get('instruction', '')).strip()
    answer = str(pair.get('answer', '')).strip()
    if len(instr) < 10 or len(answer) < 20:
        return None
    return {
        'instruction': instr,
        'output':      answer,
        'source':      f'wiki:{page}',
        'score':       1.0,
    }


def generate_pairs_for_section(section):
    """Returns list of valid pairs (may be empty on parse failure)."""
    prompt = make_generation_prompt(section)

    for attempt in range(MAX_RETRIES + 1):
        temp = 0.4 + attempt * 0.2   # raise temp on retry
        raw  = generate_raw(prompt, temp=temp)
        arr  = extract_json_array(raw)
        if arr and isinstance(arr, list):
            pairs = [validate_pair(p, section['page']) for p in arr]
            pairs = [p for p in pairs if p is not None]
            if pairs:
                return pairs

    return []

print('Generation helpers ready.')

## Main generation loop

In [ ]:
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── Resume: load already-processed section IDs ───────────────────────────────
all_pairs     = []
done_sections = set()

if OWN_FILE.exists():
    with open(OWN_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = json.loads(line)
            all_pairs.append(p)
            if 'section_id' in p:
                done_sections.add(p['section_id'])
    print(f'Resuming — {len(all_pairs)} pairs already saved, {len(done_sections)} sections done')

remaining = [s for s in all_sections if s['section_id'] not in done_sections]
print(f'Sections to process: {len(remaining)} / {len(all_sections)}')

# ── Per-page stats ────────────────────────────────────────────────────────────
page_stats = defaultdict(lambda: {'sections': 0, 'pairs': 0, 'failed': 0})

outf = open(OWN_FILE, 'a')

try:
    pbar = tqdm(total=len(remaining), desc='Wiki sections', unit='section')

    for section in remaining:
        page = section['page']
        pbar.set_description(f'{page[:30]}::{section["heading"][:20]}')

        pairs = generate_pairs_for_section(section)

        if pairs:
            for p in pairs:
                p['section_id'] = section['section_id']   # for resume tracking
                all_pairs.append(p)
                outf.write(json.dumps(p) + '\n')
            outf.flush()
            page_stats[page]['pairs'] += len(pairs)
            page_stats[page]['sections'] += 1
        else:
            page_stats[page]['failed'] += 1
            tqdm.write(f'  [WARN] No pairs for {page}::{section["heading"]}')

        pbar.set_postfix({'total': len(all_pairs), 'page': page[:20]})
        pbar.update(1)

finally:
    pbar.close()
    outf.close()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\n=== Results ===')
print(f'Total pairs generated: {len(all_pairs)}')
print()
print(f'{"Page":<45} {"Sections":>8} {"Pairs":>6} {"Failed":>7}')
print('-' * 70)
for page, s in sorted(page_stats.items()):
    print(f'{page:<45} {s["sections"]:>8} {s["pairs"]:>6} {s["failed"]:>7}')

## Quality review — sample pairs

In [ ]:
import random

# Reload all pairs from file for clean review
all_pairs_loaded = []
with open(OWN_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            all_pairs_loaded.append(json.loads(line))

print(f'Total pairs in file: {len(all_pairs_loaded)}')
print()

# Show 5 random samples
for p in random.sample(all_pairs_loaded, min(5, len(all_pairs_loaded))):
    print(f'[{p["source"]}]')
    print(f'Q: {p["instruction"]}')
    print(f'A: {p["output"][:300]}{"..." if len(p["output"]) > 300 else ""}')
    print()

## Merge into main knowledge pairs

In [ ]:
existing_keys = set()
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                p = json.loads(line)
                existing_keys.add(p.get('instruction', '')[:80])

new_count = 0
with open(PAIRS_FILE, 'a') as f:
    for pair in all_pairs_loaded:
        key = pair.get('instruction', '')[:80]
        if key not in existing_keys:
            # Remove internal metadata before merging
            clean = {k: v for k, v in pair.items() if k != 'section_id'}
            f.write(json.dumps(clean) + '\n')
            existing_keys.add(key)
            new_count += 1

print(f'Merged {new_count} new wiki pairs into {PAIRS_FILE}')
print(f'Total pairs now: {len(existing_keys)}')
print()
print('Next steps:')
print('  → Re-run notebook 04 (warm-start fine-tune) to train on wiki pairs')
print('  → Or continue to notebook 08 (synthetic data generation)')